In [ ]:
!pip install git+https://github.com/monash-emu/summer3wip.git@ab4f260aa23d01d1ee50347ea8990c54783ec074

# An SIR simulation of an infectious disease
## Intro
Here, we'll set up a relatively simple SIR-based model. It is a little more complicated than the standard three-compartment SIR model, just so that we can add in public health interventions and then adjust parameters to look at the effect of interventions. The model is intended to represent some short-lived infectious disease (such as a respiratory virus) that leads to permanent immunity following infection. We start off with some standard installs and imports, which you can ignore (including the outputs printed after the cell).

In [ ]:
import pandas as pd
from jax import numpy as jnp

pd.options.plotting.backend = "plotly"

from summer3.graph import CompartmentValues, defer, Parameter
from summer3.epi import (
    TransitionFlow,
    Stratification,
    CompartmentMap,
    CompartmentalEpiModel,
    CategoryData,
)

## Building the model
We'll add compartments to distinguish between sequential asymptomatic and symptomatic stages of infection, referring to these as `_asympt` and `_sympt`. We'll also include a vaccinated compartment, which we'll come back to below.

In [ ]:
start_time = 0
end_time = 70
model_comps = [
    "vaccinated",
    "susceptible",
    "infectious_asympt",
    "infectious_sympt",
    "recovered",
]
infect_comps = [
    "infectious_asympt",
    "infectious_sympt",
]
disease_state = Stratification("disease_state", model_comps)
humans = CompartmentMap.new(disease_state)

times = range(start_time, end_time)
sir_model = CompartmentalEpiModel(humans, times)

print(
    f"The simulation runs from time {round(start_time)} to {round(end_time)}.\n"
    f"The model compartments are called: {', '.join(model_comps)}.\n"
    f"Of these compartments, {', '.join(infect_comps)} are considered equally infectious."
)

## Data
Here are some synthetic data, that were created by running the model, getting the outputs, adding noise and rounding.

In [ ]:
data = pd.Series(
    {
        20.0: 187.0,
        22.0: 302.0,
        24.0: 982.0,
        26.0: 1383.0,
        28.0: 2203.0,
        30.0: 4304.0,
        32.0: 5479.0,
        34.0: 12945.0,
        36.0: 11310.0,
        38.0: 14238.0,
        40.0: 9262.0,
        42.0: 3677.0,
        44.0: 2899.0,
        46.0: 2149.0,
        48.0: 1443.0,
        50.0: 1049.0,
    },
)

## Infection process
Next, we'll specify that the force of infection (which is the rate at which susceptible persons are infected)
is related to the prevalence of infection (scaled by a parameter).
This assumption is referred to as frequency dependent transmission.
It isn't the only way to simulate infection, but is one of the commonest.

In [ ]:
def frequency_dependent_transmission(
    compartment_values, infectious_comps, contact_rate
):
    infectious_pop = compartment_values.query(infectious_comps).sum()
    total_pop = compartment_values.sum()
    infectious_prevalence = infectious_pop / total_pop
    return contact_rate * infectious_prevalence

## Interventions
### Face masks
Let's implement population-wide use of face masks into our simple model. Rather than the infection rate being determined by the contact rate parameter alone, we'll adjust this based on the proportion of people wearing the masks ("coverage") and the efficacy of wearing the mask on preventing transmission in those who are wearing them.

In [ ]:
mask_cov = Parameter("face_mask_coverage", 0.0)
mask_effic = Parameter("face_mask_efficacy", 0.0)
contact_rate = Parameter("contact_rate", 0.0)
inf_rate = contact_rate * (1.0 - mask_cov * mask_effic)
source = "susceptible"
dest = "infectious_asympt"

force_of_infection = defer(frequency_dependent_transmission)(
    CompartmentValues,
    disease_state[infect_comps],
    inf_rate,
)

infection = TransitionFlow(
    "infection",
    disease_state["susceptible"],
    disease_state["infectious_asympt"],
    force_of_infection,
)
sir_model.add_flow(infection)

print(
    f"The process of infection transitions people from the {source} compartment to the {dest} compartment.\n"
    f"The rate of infection is determined by the coverage '{mask_cov.key}' and efficacy '{mask_effic.key}' of face masks.\n"
    "The product of these two parameters determine how much face masks reduce transmission by."
)

### Vaccination
Let's also allow that vaccination can reduce the rate at which people are infected, but only apply this to the vaccinated population. We included a vaccinated compartment earlier, so we'll have to apply an infection process to this compartment too, but adjust the rate at which people from this compartment are infected according to the efficacy of the vaccine being used. We'll only see an effect from this if we start some of the population off in the vaccinated compartment, of course.

In [ ]:
vacc_effic = Parameter("vacc_efficacy", 0.0)
vacc_infection_rate = force_of_infection * (1.0 - vacc_effic)

infection_vacc = TransitionFlow(
    "infection_vacc",
    disease_state["vaccinated"],
    disease_state["infectious_asympt"],
    vacc_infection_rate,
)
sir_model.add_flow(infection_vacc)

print(
    "Vaccination reduces the rate of infection for the vaccinated population "
    f"according to the efficacy of the vaccine, which is given by the parameter {vacc_effic.key}."
)

### Case isolation
We'll now look at incorporating the effect of case isolation for infected people with symptoms. Because we have two infectious compartments in series, if we want the average time infectious (in the absence of case isolation) to be the reciprocal of the recovery rate, we'll have to double the rate of transition between the compartments. Once we've done that, we can add an additional rate at which people with symptoms effectively remove themselves from the infectious population, which we can refer to as case isolation. Because this only applies to the compartment with symptoms (i.e. the second half of the infectious period), we can add this on to the rate of transition from `infectious_sympt` to `recovered`.

If we're going to work out the rate of case notifications in the model, we'll also have to track some rate of transition between compartments because this is an incident (rather than prevalent) quantity. So in the model, this is a rate of movement between compartments rather than the size of a compartment.

In [ ]:
progression_rate = Parameter("recovery_rate", 0.0) * 2.0
progression = TransitionFlow(
    "progression",
    disease_state["infectious_asympt"],
    disease_state["infectious_sympt"],
    progression_rate,
)
sir_model.add_flow(progression)

isolation_rate = Parameter("isolation_rate", 0.0)
resolve_sympt_rate = progression_rate + isolation_rate
resolution = TransitionFlow(
    "resolution",
    disease_state["infectious_sympt"],
    disease_state["recovered"],
    resolve_sympt_rate,
)
sir_model.add_flow(resolution)

print(
    "For case isolation, we add an additional flow that increases "
    f"the rate at which people move from the {source} compartment "
    f"to the {dest} compartment. "
)

## Preparing the model
We'll start with a total population of one ninth of 7 million, based on Victoria's approximate total population of 7 million divided by nine LPHUs.

In [ ]:
total_population = 7e6 / 9.0
infectious_seed = 1.0

vacc_prop = 0.3

suscept_pop = total_population - infectious_seed

suscept_vacc = vacc_prop * suscept_pop
suscept_unvacc = (1.0 - vacc_prop) * suscept_pop

init_pops = CategoryData(
    disease_state.categories(),
    jnp.array(([suscept_vacc, suscept_unvacc, infectious_seed, 0.0, 0.0])),
)
sir_model.set_initial_population(init_pops)
print(
    f"The total population simulated to be {total_population} persons.\n"
    f"This population is seeded with {infectious_seed} infectious persons to trigger the epidemic. "
)

## Running the model with interventions
Now we have a model that can run three different interventions, and any combination of those three interventions together. This cell is for you to experiment with different values for the five different intervention-related input parameters. See how they affect the epidemic and whether the results are consistent with what you expected.

Note that if you want to change any of the structures of the model above, you will probably need to re-run all the preceding cells, but if you're happy with the structure, you should be able to adjust the parameters and re-run in this cell alone. If things seem to have got tangled, re-run the notebook from the start to make sure the code has all been run in the expected order.

In [ ]:
infection_parameters = {
    "contact_rate": 0.5,
    "recovery_rate": 0.2,
}
intervention_parameters = {
    "face_mask_coverage": 0.0,
    "face_mask_efficacy": 0.0,
    "isolation_rate": 0.0,
    "vacc_efficacy": 0.0,
}
case_detection_prop = 0.2
results = sir_model.run(infection_parameters | intervention_parameters)
incidence = pd.DataFrame(results["flows"]["infection"].data) + pd.DataFrame(
    results["flows"]["infection_vacc"].data
)
notifications = incidence * case_detection_prop
notifications.columns = ["model"]
fig = notifications.plot()
fig.add_scatter(x=data.index, y=data, mode="markers", name="data")